# 🎓 Smart Major & Campus Life Matcher — LangChain Multi-Agent Demo

An end-to-end multi-agent workflow powered by **LangChain (`create_agent`)** and **Ollama (`gemma4:e2b`)** for **University Admissions Events & Open House Days**.

### **Architecture**:
Uses `create_agent` factory functions combining `ChatOllama` LLM, `ChatPromptTemplate`, and `PydanticOutputParser` for structured JSON output across 4 specialized agents.

```
[Student Raw Input]
       │
       ▼
[Agent 1: create_agent(Extractor)] ──► StudentProfile (Pydantic)
       │
       ├────────────────────────────────┐
       ▼                                ▼
[Agent 2: create_agent(Major Matcher)]   [Agent 3: create_agent(Club Matcher)]
(Evaluates majors.html dataset)          (Evaluates clubs.html dataset)
       │                                │
       └────────────────────────────────┘
       │
       ▼
[Agent 4: create_agent(Synthesizer)] ──► 4-Year Campus Preview
```

In [ ]:
import json
from pydantic import BaseModel, Field
from typing import List, Dict

from data_loader import MAJORS_DATA, CLUBS_DATA, get_majors_summary, get_clubs_summary
from app import create_agent, StudentProfile, MajorMatch, ClubMatch, CampusRoadmap, run_agentic_pipeline

print(f"✓ Loaded dataset: {len(MAJORS_DATA)} majors & {len(CLUBS_DATA)} campus clubs.")

## 1. LangChain Agent Factory Function (`create_agent`)

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

def create_agent(llm, system_prompt: str, pydantic_schema=None):
    """
    Factory function creating an LLM Agent chain using LangChain ChatPromptTemplate,
    ChatOllama model, and optional PydanticOutputParser for structured JSON outputs.
    """
    if pydantic_schema:
        parser = PydanticOutputParser(pydantic_object=pydantic_schema)
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt + "\n\nCRITICAL: You MUST respond strictly in valid JSON matching this format instructions:\n{format_instructions}"),
            ("user", "{input_text}")
        ]).partial(format_instructions=parser.get_format_instructions())
        chain = prompt | llm | parser
    else:
        prompt = ChatPromptTemplate.from_messages([
            ("system", system_prompt),
            ("user", "{input_text}")
        ])
        chain = prompt | llm
    return chain

## 2. Test Pipeline Execution

In [ ]:
DEMO_INPUT = "I love robotics, python programming, building microcontrollers, and want to study artificial intelligence to start a software company."

# Execute pipeline targeted at local Ollama model gemma4:e2b
markdown_report, agent1_json, agent2_json, agent3_json, telemetry = run_agentic_pipeline(
    student_raw_input=DEMO_INPUT,
    model_name="gemma4:e2b",
    base_url="http://localhost:11434"
)

print("=== TELEMETRY & LOGS ===")
print(telemetry)

print("\n=== AGENT 1 OUTPUT (StudentProfile) ===")
print(agent1_json)

## 3. Launch Interactive Gradio Web UI

In [ ]:
from app import create_gradio_app

# Launch Gradio Web App UI
demo_app = create_gradio_app()
demo_app.launch(share=False)